# Brain Age Prediction - Google Colab

This notebook implements brain age prediction using VGG16 features extracted from MRI scans.

**Two fusion strategies are compared:**
- **Early Fusion**: Aggregate triplet features first, then train single SVR
- **Late Fusion**: Train separate SVR per triplet, then combine predictions

**Instructions:**
1. Go to Runtime > Change runtime type > Select **GPU** (T4 recommended)
2. Upload your IXI data to Google Drive
3. Run all cells in order

## 1. Setup & GPU Check

In [ ]:
# Check GPU availability
import tensorflow as tf

print(f"TensorFlow version: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs available: {len(gpus)}")

if gpus:
    for gpu in gpus:
        print(f"  - {gpu}")
    # Enable memory growth to avoid OOM
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    print("WARNING: No GPU found! Go to Runtime > Change runtime type > GPU")

In [ ]:
# Install required packages
!pip install -q nibabel opencv-python-headless xlrd openpyxl

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ============================================
# MODIFY THESE PATHS TO MATCH YOUR DRIVE STRUCTURE
# ============================================
DATA_BASE_DIR = '/content/drive/MyDrive/Data_IXI/June'
IXI_IMAGE_DIR = f'{DATA_BASE_DIR}/IXI-T1'
IXI_DEMOGRAPHICS_FILE = f'{DATA_BASE_DIR}/IXI.xls'

import os
print(f"\nChecking data paths...")
print(f"Image dir exists: {os.path.exists(IXI_IMAGE_DIR)}")
print(f"Demographics file exists: {os.path.exists(IXI_DEMOGRAPHICS_FILE)}")

if os.path.exists(IXI_IMAGE_DIR):
    nii_files = [f for f in os.listdir(IXI_IMAGE_DIR) if f.endswith(('.nii', '.nii.gz'))]
    print(f"Found {len(nii_files)} NIfTI files")

## 3. Import Libraries & Load CNN Model

In [ ]:
import os
import numpy as np
import pandas as pd
import nibabel as nib
import cv2
from sklearn import svm
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from scipy.stats import pearsonr
import time

from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.models import Model

# Load VGG16 model
print("Loading VGG16 model...")
cnn_base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
FEATURE_EXTRACTOR_TARGET_LAYER = 'block5_pool'

# Create feature extractor
feature_extractor = Model(
    inputs=cnn_base_model.input,
    outputs=cnn_base_model.get_layer(FEATURE_EXTRACTOR_TARGET_LAYER).output
)
print(f"Feature extractor output shape: {feature_extractor.output_shape}")
print("Model loaded successfully!")

## 4. Load Demographics & Create Data Split

In [ ]:
# Load demographics
print("Loading demographics...")
df_demographics = pd.read_excel(IXI_DEMOGRAPHICS_FILE, sheet_name=0)
df_demographics_cleaned = df_demographics.dropna(subset=['AGE']).copy()
df_demographics_cleaned['IXI_ID'] = df_demographics_cleaned['IXI_ID'].astype(int)
print(f"Demographics loaded: {len(df_demographics_cleaned)} subjects with valid ages")

# Get list of .nii files and extract subject IDs
ixi_nii_files_all = [f for f in os.listdir(IXI_IMAGE_DIR) if f.endswith('.nii') or f.endswith('.nii.gz')]
print(f"Found {len(ixi_nii_files_all)} NIfTI files")

# Build mapping of subject ID to filename
file_mapping = {}
for f in ixi_nii_files_all:
    ixi_id = int(f.split('-')[0].replace('IXI', ''))
    file_mapping[ixi_id] = f

# Get subjects with both MRI and demographics
available_ids = set(file_mapping.keys()) & set(df_demographics_cleaned['IXI_ID'].values)
available_ids = sorted(list(available_ids))
print(f"Matched subjects (MRI + demographics): {len(available_ids)}")

# Get ages for available subjects
df_available = df_demographics_cleaned[df_demographics_cleaned['IXI_ID'].isin(available_ids)]
df_available = df_available.set_index('IXI_ID').loc[available_ids].reset_index()

Y_all_ages = df_available['AGE'].values
I_all_ids = df_available['IXI_ID'].values

print(f"Age range: {Y_all_ages.min():.1f} - {Y_all_ages.max():.1f} years")
print(f"Mean age: {Y_all_ages.mean():.1f} +/- {Y_all_ages.std():.1f} years")

In [ ]:
# Data splitting (80-20)
n_subjects = len(available_ids)
indices = np.arange(n_subjects)
np.random.seed(42)  # For reproducibility
np.random.shuffle(indices)

split_idx = int(0.8 * n_subjects)
train_indices = indices[:split_idx]
test_indices = indices[split_idx:]

Y_train = Y_all_ages[train_indices]
Y_test = Y_all_ages[test_indices]
train_ids = I_all_ids[train_indices]
test_ids = I_all_ids[test_indices]

print(f"Training samples: {len(train_indices)}")
print(f"Test samples: {len(test_indices)}")

# Plot age distributions
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(Y_all_ages, bins=30, alpha=0.5, label='All', color='gray')
ax.hist(Y_train, bins=30, alpha=0.7, label='Train', color='blue')
ax.hist(Y_test, bins=30, alpha=0.7, label='Test', color='green')
ax.set_xlabel('Age (years)')
ax.set_ylabel('Count')
ax.set_title('Age Distribution')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Feature Extraction (GPU Accelerated)

This extracts both:
- **Aggregated features** (mean of triplets) for Early Fusion
- **Per-triplet features** for Late Fusion

In [ ]:
def extract_features_with_triplets(file_paths, feature_extractor, batch_size=8, num_slices=120):
    """
    Extract features for multiple subjects using GPU batching.
    Returns both aggregated features (for early fusion) and per-triplet features (for late fusion).
    """
    all_aggregated = []
    all_triplets = []
    input_size = (224, 224)
    
    for file_path in tqdm(file_paths, desc="Extracting features"):
        try:
            # Load NIfTI
            nii_img = nib.load(file_path)
            mri_vol = nii_img.get_fdata()[:, :, :num_slices]
            
            # Create triplets
            num_slices_vol = min(mri_vol.shape[2], num_slices)
            total_triplets = num_slices_vol // 3
            
            # Prepare batch of triplets
            triplet_batch = []
            for k in range(total_triplets):
                idx = k * 3
                s1, s2, s3 = mri_vol[:, :, idx], mri_vol[:, :, idx+1], mri_vol[:, :, idx+2]
                triplet_img = np.stack((s1, s2, s3), axis=-1)
                resized = cv2.resize(triplet_img.astype(np.float32), input_size, interpolation=cv2.INTER_LINEAR)
                triplet_batch.append(resized)
            
            # Process triplets in batches on GPU
            triplet_batch = np.array(triplet_batch)
            triplet_batch = preprocess_input(triplet_batch)
            
            # Extract features (GPU accelerated)
            features = feature_extractor.predict(triplet_batch, batch_size=batch_size, verbose=0)
            features_flat = features.reshape(features.shape[0], -1)  # (num_triplets, 25088)
            
            # Store per-triplet features (for late fusion)
            all_triplets.append(features_flat)
            
            # Aggregate (mean across triplets) for early fusion
            aggregated = np.mean(features_flat, axis=0)
            all_aggregated.append(aggregated)
            
        except Exception as e:
            print(f"Error processing {file_path}: {e}")
            all_aggregated.append(np.zeros(25088))
            all_triplets.append(np.zeros((40, 25088)))  # Assume 40 triplets
    
    return np.array(all_aggregated), all_triplets

In [ ]:
# Get file paths for train and test
train_paths = [os.path.join(IXI_IMAGE_DIR, file_mapping[sid]) for sid in train_ids]
test_paths = [os.path.join(IXI_IMAGE_DIR, file_mapping[sid]) for sid in test_ids]

print(f"Extracting features for {len(train_paths)} training subjects...")
start_time = time.time()
X_train_agg, X_train_triplets = extract_features_with_triplets(train_paths, feature_extractor, batch_size=16)
print(f"Training aggregated features shape: {X_train_agg.shape}")
print(f"Training triplet features: {len(X_train_triplets)} subjects, first subject shape: {X_train_triplets[0].shape}")

print(f"\nExtracting features for {len(test_paths)} test subjects...")
X_test_agg, X_test_triplets = extract_features_with_triplets(test_paths, feature_extractor, batch_size=16)
print(f"Test aggregated features shape: {X_test_agg.shape}")

elapsed = time.time() - start_time
print(f"\nTotal extraction time: {elapsed/60:.1f} minutes")

## 6. Early Fusion

In [ ]:
print("\n" + "="*60)
print("EARLY FUSION")
print("="*60)

# Optional: Normalize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_agg)
X_test_scaled = scaler.transform(X_test_agg)

# Train SVR (parameters from original MATLAB code)
print("Training SVR model...")
early_svr = svm.SVR(kernel='linear', C=98, epsilon=0.064)
early_svr.fit(X_train_scaled, Y_train)
print("Training complete!")

# Predict
Y_pred_early = early_svr.predict(X_test_scaled)

# Evaluate
mae_early = np.mean(np.abs(Y_test - Y_pred_early))
rmse_early = np.sqrt(np.mean((Y_test - Y_pred_early) ** 2))
corr_early, _ = pearsonr(Y_test, Y_pred_early)

print(f"\nEarly Fusion Results:")
print(f"  MAE:         {mae_early:.2f} years")
print(f"  RMSE:        {rmse_early:.2f} years")
print(f"  Correlation: {corr_early:.4f}")

## 7. Late Fusion

In [ ]:
print("\n" + "="*60)
print("LATE FUSION")
print("="*60)
print("Training weak learners (one SVR per triplet modality)...")

# Get number of triplets
num_triplets = X_train_triplets[0].shape[0]
num_train = len(X_train_triplets)
num_test = len(X_test_triplets)

print(f"  Number of triplet modalities: {num_triplets}")
print(f"  Training subjects: {num_train}, Test subjects: {num_test}")

# Train one SVR per triplet and get predictions
weak_predictions = np.zeros((num_triplets, num_test))

for t in tqdm(range(num_triplets), desc="Training weak learners"):
    # Get features for triplet t from all subjects
    X_train_t = np.array([X_train_triplets[j][t, :] for j in range(num_train)])
    X_test_t = np.array([X_test_triplets[j][t, :] for j in range(num_test)])
    
    # Train SVR for this triplet
    weak_svr = svm.SVR(kernel='linear', C=1.0, epsilon=0.1)
    weak_svr.fit(X_train_t, Y_train)
    
    # Predict on test data
    weak_predictions[t, :] = weak_svr.predict(X_test_t)

print(f"\nWeak predictions shape: {weak_predictions.shape}")

# Combine predictions using mean
Y_pred_late = np.mean(weak_predictions, axis=0)

# Evaluate Late Fusion
mae_late = np.mean(np.abs(Y_test - Y_pred_late))
rmse_late = np.sqrt(np.mean((Y_test - Y_pred_late) ** 2))
corr_late, _ = pearsonr(Y_test, Y_pred_late)

print(f"\nLate Fusion Results:")
print(f"  MAE:         {mae_late:.2f} years")
print(f"  RMSE:        {rmse_late:.2f} years")
print(f"  Correlation: {corr_late:.4f}")

## 8. Results Comparison

In [ ]:
print("\n" + "="*60)
print("COMPARISON SUMMARY")
print("="*60)
print(f"{'Method':<25} {'MAE':>10} {'RMSE':>10} {'Corr':>10}")
print("-"*60)
print(f"{'Early Fusion (SVR)':<25} {mae_early:>10.2f} {rmse_early:>10.2f} {corr_early:>10.4f}")
print(f"{'Late Fusion (Mean)':<25} {mae_late:>10.2f} {rmse_late:>10.2f} {corr_late:>10.4f}")
print("="*60)

# Determine best method
if mae_early < mae_late:
    print(f"\n>>> Best method: Early Fusion (lower MAE by {mae_late - mae_early:.2f} years)")
else:
    print(f"\n>>> Best method: Late Fusion (lower MAE by {mae_early - mae_late:.2f} years)")

## 9. Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Early Fusion scatter plot
ax = axes[0, 0]
ax.scatter(Y_test, Y_pred_early, alpha=0.6, edgecolors='black', linewidth=0.5, color='blue')
ax.plot([Y_test.min(), Y_test.max()], [Y_test.min(), Y_test.max()], 'r--', lw=2)
ax.set_xlabel('Chronological Age (years)', fontsize=12)
ax.set_ylabel('Predicted Brain Age (years)', fontsize=12)
ax.set_title(f'Early Fusion\nMAE={mae_early:.2f}, r={corr_early:.3f}', fontsize=14)
ax.grid(True, alpha=0.3)

# Late Fusion scatter plot
ax = axes[0, 1]
ax.scatter(Y_test, Y_pred_late, alpha=0.6, edgecolors='black', linewidth=0.5, color='green')
ax.plot([Y_test.min(), Y_test.max()], [Y_test.min(), Y_test.max()], 'r--', lw=2)
ax.set_xlabel('Chronological Age (years)', fontsize=12)
ax.set_ylabel('Predicted Brain Age (years)', fontsize=12)
ax.set_title(f'Late Fusion\nMAE={mae_late:.2f}, r={corr_late:.3f}', fontsize=14)
ax.grid(True, alpha=0.3)

# Early Fusion error distribution
ax = axes[1, 0]
errors_early = Y_pred_early - Y_test
ax.hist(errors_early, bins=30, alpha=0.7, color='blue', edgecolor='black')
ax.axvline(x=0, color='red', linestyle='--', linewidth=2)
ax.axvline(x=np.mean(errors_early), color='orange', linestyle='-', linewidth=2, label=f'Mean={np.mean(errors_early):.2f}')
ax.set_xlabel('Prediction Error (years)', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Early Fusion Error Distribution', fontsize=14)
ax.legend()

# Late Fusion error distribution
ax = axes[1, 1]
errors_late = Y_pred_late - Y_test
ax.hist(errors_late, bins=30, alpha=0.7, color='green', edgecolor='black')
ax.axvline(x=0, color='red', linestyle='--', linewidth=2)
ax.axvline(x=np.mean(errors_late), color='orange', linestyle='-', linewidth=2, label=f'Mean={np.mean(errors_late):.2f}')
ax.set_xlabel('Prediction Error (years)', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Late Fusion Error Distribution', fontsize=14)
ax.legend()

plt.tight_layout()
plt.savefig('brain_age_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nFigure saved!")

## 10. Save Results

In [ ]:
# Create results DataFrame
results_df = pd.DataFrame({
    'Subject_ID': test_ids,
    'True_Age': Y_test,
    'Predicted_EarlyFusion': Y_pred_early,
    'Error_EarlyFusion': Y_pred_early - Y_test,
    'Predicted_LateFusion': Y_pred_late,
    'Error_LateFusion': Y_pred_late - Y_test
})

# Save predictions
results_df.to_csv('brain_age_predictions.csv', index=False)
print("Predictions saved to brain_age_predictions.csv")

# Save summary
summary_df = pd.DataFrame({
    'Metric': ['MAE', 'RMSE', 'Correlation', 'Num_Train', 'Num_Test'],
    'Early_Fusion': [mae_early, rmse_early, corr_early, len(train_ids), len(test_ids)],
    'Late_Fusion': [mae_late, rmse_late, corr_late, len(train_ids), len(test_ids)]
})
summary_df.to_csv('brain_age_summary.csv', index=False)
print("Summary saved to brain_age_summary.csv")

# Display sample results
print("\nSample predictions:")
results_df.head(10)

In [ ]:
# Display summary
print("\nSummary:")
summary_df

## 11. Download Results

In [ ]:
from google.colab import files

# Download result files
files.download('brain_age_predictions.csv')
files.download('brain_age_summary.csv')
files.download('brain_age_results.png')

---
## Workflow Complete!

**Results Summary:**

| Method | MAE | RMSE | Correlation |
|--------|-----|------|-------------|
| Early Fusion | See above | See above | See above |
| Late Fusion | See above | See above | See above |

**Files saved:**
- `brain_age_predictions.csv` - Individual subject predictions (both methods)
- `brain_age_summary.csv` - Summary metrics
- `brain_age_results.png` - Visualization plots